In [1]:
import pandas as pd
from scipy.stats import ttest_rel, wilcoxon
import numpy as np

In [3]:
df_off = pd.read_csv('converging_timing_triangulation_off.csv')
df_on  = pd.read_csv('converging_timing_triangulation_on.csv')

In [4]:
df_off.rename(columns={'time': 'off'}, inplace=True)
df_on.rename(columns={'time': 'on'}, inplace=True)

In [5]:
df = pd.merge(df_off, df_on, on='map_name')

In [6]:
df

,map_name,off,on
0,egypt_population_by_governate_2017,0.933,0.936
1,algeria_population_by_wilaya_2022,1.157,0.995
2,austria_population_by_state_2020,0.226,0.228
3,usa_population_by_state_2020,0.509,0.523
4,russia_population_by_federal_subject_2010,1.293,1.575
5,metropolitan_france_population_by_departement_...,0.465,0.458
6,croatia_covid_cases_by_county_2022,0.322,0.290
7,brazil_population_by_state_2021,0.548,0.543
8,belgium_population_by_region_2022,0.103,0.099
9,switzerland_gdp_in_billion_chf_by_canton_2019,0.425,0.419


In [7]:
for col in ['off', 'on']:
    mean_val   = df[col].mean()
    median_val = df[col].median()
    std_val    = df[col].std()
    print(f"{col} — mean: {mean_val:.6f}, median: {median_val:.6f}, std: {std_val:.6f}")

off — mean: 2.108219, median: 0.454000, std: 6.378338
on — mean: 2.193562, median: 0.463500, std: 6.777812


In [ ]:
stat, p = ttest_rel(df['off'], df['on'])
print(f"paired t-statistic = {stat:.6f}, p-value = {p:.6f}")

if p < 0.05:
    print("The difference in means is statistically significant (p < 0.05).")
else:
    print("No statistically significant difference (p >= 0.05).")

paired t‑statistic = -1.146893, p‑value = 0.260201
No statistically significant difference (p >= 0.05).


In [14]:
# 1) Non-parametric paired test (Wilcoxon signed‑rank)
stat_w, p_w = wilcoxon(df['off'], df['on'])
print(f"Wilcoxon signed-rank -> stat: {stat_w:.4f}, p-value: {p_w:.6f}")

# 2) Sensitivity: drop the single largest "off" and re-run paired t-test
out_idx = df['off'].idxmax()
df_trim = df.drop(index=out_idx)
stat_t2, p_t2 = ttest_rel(df_trim['off'], df_trim['on'])
print(f"Paired t-test w/o top-off map -> t: {stat_t2:.4f}, p: {p_t2:.6f}")

# 3) Bootstrap 95% CI on mean (off – on)
n_boot = 10_000
boot_diffs = np.empty(n_boot)
arr_off, arr_on = df['off'].values, df['on'].values
for i in range(n_boot):
    idxs = np.random.randint(0, len(arr_off), len(arr_off))
    boot_diffs[i] = np.mean(arr_off[idxs] - arr_on[idxs])
ci_lower, ci_upper = np.percentile(boot_diffs, [2.5, 97.5])
print(f"Bootstrap 95% CI of mean(off-on): [{ci_lower:.6f}, {ci_upper:.6f}]")

Wilcoxon signed-rank -> stat: 253.0000, p-value: 0.846460
Paired t-test w/o top-off map -> t: -0.6356, p: 0.529861
Bootstrap 95% CI of mean(off-on): [-0.254125, 0.016439]
